# Dataset Overview

## Materials Selected
1. NLP Course Material
2. Cloud Computing Course Material
3. Pakistan Studies Course Material
4. "The Sealed Nectar" - Award-winning biography of Prophet Muhammad

## Preprocessing
Each lecture file (2000-5000 words) was manually divided into contextually coherent parts of ~500 words to maintain semantic integrity.

# Chunking Results

Additional chunking applied using **intfloat/e5-base** tokenizer

## Dataset Statistics
- **Initial manual chunks:** 568 files
- **After automatic chunking:** 704 files

In [ ]:
import os
import re
import json
import hashlib
from typing import List, Dict, Tuple, Optional
from transformers import AutoTokenizer
from pathlib import Path
import nltk
from collections import Counter
import numpy as np

try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')

class EnhancedDocumentChunker:
    def __init__(self, 
                 model_name: str = "intfloat/e5-base",
                 chunk_size: int = 512,   # limit of tokens of each chunk
                 overlap: int = 100,
                 min_chunk_size: int = 200,   # min limit  
                 max_chunk_size: int = 800):  # max limit
        """Initialize document chunker with semantic splitting"""
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.chunk_size = chunk_size
        self.overlap = overlap
        self.min_chunk_size = min_chunk_size
        self.max_chunk_size = max_chunk_size
        self.chunk_counter = 0
        
        self.category_configs = {
            'ICC': {'technical_weight': 1.5, 'context_boost': True},
            'NLP': {'technical_weight': 1.8, 'context_boost': True},
            'Pakistan Studies': {'narrative_weight': 1.2, 'context_boost': False},
            'Sealed Nectar': {'narrative_weight': 1.0, 'context_boost': False}
        }
        
    def estimate_tokens(self, text: str) -> int:
        """Estimate token count using tokenizer"""
        return len(self.tokenizer.tokenize(text))
    
    def extract_enhanced_keywords(self, filename: str, text_sample: str = "") -> Dict[str, any]:
        """Extract keywords from filename and content"""
        name_without_ext = filename.replace('.txt', '')
        separators = ['_', '-', ' ', '.', '(', ')', '[', ']']
        filename_keywords = [name_without_ext]
        
        for sep in separators:
            new_keywords = []
            for keyword in filename_keywords:
                new_keywords.extend(keyword.split(sep))
            filename_keywords = new_keywords
        
        filename_keywords = [
            re.sub(r'\s+', ' ', kw.strip().lower()) 
            for kw in filename_keywords 
            if len(kw.strip()) > 2 and kw.strip().lower() not in {'txt', 'file', 'doc', 'the', 'and', 'for', 'with'}
        ]
        
        content_keywords = []
        if text_sample:
            technical_terms = re.findall(r'\b[A-Z][A-Za-z]*(?:\s+[A-Z][A-Za-z]*)*\b', text_sample)
            acronyms = re.findall(r'\b[A-Z]{2,}\b', text_sample)
            quoted_terms = re.findall(r'["\']([^"\']{3,30})["\']', text_sample)
            
            content_keywords = technical_terms + acronyms + quoted_terms
            content_keywords = [kw.lower().strip() for kw in content_keywords if len(kw.strip()) > 2]
        
        return {
            'filename_keywords': list(set(filename_keywords)),
            'content_keywords': list(set(content_keywords)),
            'all_keywords': list(set(filename_keywords + content_keywords)),
            'keyword_string': ' '.join(set(filename_keywords + content_keywords))
        }
    
    def smart_text_splitting(self, text: str) -> List[str]:
        """Split text respecting semantic boundaries"""
        text = re.sub(r'\n\s*\n\s*\n+', '\n\n', text)
        text = re.sub(r'[ \t]+', ' ', text)
        text = text.strip()
        
        sections = re.split(r'\n\s*\n', text)
        
        all_units = []
        for section in sections:
            if not section.strip():
                continue
                
            try:
                sentences = nltk.sent_tokenize(section.strip())
                all_units.extend(sentences)
            except:
                sentences = re.split(r'(?<=[.!?])\s+', section.strip())
                all_units.extend([s.strip() for s in sentences if s.strip()])
        
        return [unit for unit in all_units if len(unit.strip()) > 20]
    
    def create_contextual_overlap(self, previous_chunk: str, overlap_size: int) -> str:
        """Create overlap preserving sentence boundaries"""
        if not previous_chunk:
            return ""
        
        sentences = nltk.sent_tokenize(previous_chunk)
        overlap_text = ""
        current_tokens = 0
        
        for sentence in reversed(sentences):
            sentence_tokens = self.estimate_tokens(sentence)
            if current_tokens + sentence_tokens <= overlap_size:
                overlap_text = sentence + " " + overlap_text
                current_tokens += sentence_tokens
            else:
                break
        
        return overlap_text.strip()
    
    def create_enhanced_chunks(self, text: str, source_info: Dict) -> List[Dict]:
        """Create semantically aware chunks with metadata"""
        text_units = self.smart_text_splitting(text)
        if not text_units:
            return []
        
        content_sample = ' '.join(text_units[:3])
        keyword_data = self.extract_enhanced_keywords(source_info['file_name'], content_sample)
        category_config = self.category_configs.get(source_info['category'], {})
        
        chunks = []
        current_chunk_units = []
        current_tokens = 0
        previous_chunk_text = ""
        
        for i, unit in enumerate(text_units):
            unit_tokens = self.estimate_tokens(unit)
            
            if (current_tokens + unit_tokens > self.chunk_size and 
                current_tokens >= self.min_chunk_size):
                
                chunk_text = ' '.join(current_chunk_units)
                
                if previous_chunk_text and self.overlap > 0:
                    overlap_text = self.create_contextual_overlap(previous_chunk_text, self.overlap)
                    if overlap_text:
                        chunk_text = overlap_text + " " + chunk_text
                
                chunk_data = self.create_chunk_metadata(
                    chunk_text, source_info, keyword_data, category_config, i
                )
                chunks.append(chunk_data)
                
                previous_chunk_text = chunk_text
                current_chunk_units = [unit]
                current_tokens = unit_tokens
                
            else:
                current_chunk_units.append(unit)
                current_tokens += unit_tokens
        
        if current_chunk_units and current_tokens >= self.min_chunk_size:
            chunk_text = ' '.join(current_chunk_units)
            
            if previous_chunk_text and self.overlap > 0:
                overlap_text = self.create_contextual_overlap(previous_chunk_text, self.overlap)
                if overlap_text:
                    chunk_text = overlap_text + " " + chunk_text
            
            chunk_data = self.create_chunk_metadata(
                chunk_text, source_info, keyword_data, category_config, len(text_units)
            )
            chunks.append(chunk_data)
        
        return chunks
    
    def create_chunk_metadata(self, chunk_text: str, source_info: Dict, 
                            keyword_data: Dict, category_config: Dict, position: int) -> Dict:
        """Create comprehensive chunk metadata"""
        chunk_hash = hashlib.md5(chunk_text.encode()).hexdigest()[:12]
        
        enhanced_text = chunk_text
        if category_config.get('context_boost'):
            enhanced_text = f"[{source_info['category']}] {chunk_text}"
        
        word_count = len(chunk_text.split())
        sentence_count = len(nltk.sent_tokenize(chunk_text))
        token_count = self.estimate_tokens(chunk_text)
        
        chunk_data = {
            'chunk_id': self.chunk_counter,
            'chunk_hash': chunk_hash,
            'text': chunk_text,
            'enhanced_text': enhanced_text,
            'category': source_info['category'],
            'source_file': source_info['file_path'],
            'file_name': source_info['file_name'],
            'position_in_document': position,
            'filename_keywords': keyword_data['filename_keywords'],
            'content_keywords': keyword_data['content_keywords'],
            'all_keywords': keyword_data['all_keywords'],
            'keyword_string': keyword_data['keyword_string'],
            'token_count': token_count,
            'word_count': word_count,
            'sentence_count': sentence_count,
            'technical_weight': category_config.get('technical_weight', 1.0),
            'narrative_weight': category_config.get('narrative_weight', 1.0),
            'chunk_density': word_count / token_count if token_count > 0 else 0,
            'avg_sentence_length': word_count / sentence_count if sentence_count > 0 else 0,
        }
        
        self.chunk_counter += 1
        return chunk_data
    
    def process_document_folders(self, base_path: str) -> Tuple[List[Dict], Dict]:
        """Process all document folders"""
        folder_categories = {
            'icc_text_files': 'ICC',
            'NLP_text_files': 'NLP',
            'pakSt_text_files': 'Pakistan Studies',
            'Sealed_nectar_text_files': 'Sealed Nectar'
        }
        
        all_chunks = []
        detailed_stats = {
            'processing_stats': {},
            'keyword_stats': {},
            'quality_metrics': {},
            'chunk_size_distribution': []
        }
        
        print("🚀 Enhanced Document Processing Started")
        print("=" * 70)
        
        for folder_name, category in folder_categories.items():
            folder_path = os.path.join(base_path, folder_name)
            
            if not os.path.exists(folder_path):
                print(f"⚠️  Folder not found: {folder_name}")
                continue
            
            print(f"📁 Processing: {category} ({folder_name})")
            
            folder_chunks = []
            folder_stats = {'files_processed': 0, 'empty_files': 0, 'error_files': 0}
            folder_keywords = set()
            chunk_sizes = []
            
            for file_name in os.listdir(folder_path):
                if file_name.endswith('.txt'):
                    file_path = os.path.join(folder_path, file_name)
                    
                    try:
                        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                            content = f.read()
                        
                        if not content.strip():
                            folder_stats['empty_files'] += 1
                            continue
                        
                        source_info = {
                            'category': category,
                            'file_path': file_path,
                            'file_name': file_name
                        }
                        
                        file_chunks = self.create_enhanced_chunks(content, source_info)
                        folder_chunks.extend(file_chunks)
                        folder_stats['files_processed'] += 1
                        
                        for chunk in file_chunks:
                            folder_keywords.update(chunk['all_keywords'])
                            chunk_sizes.append(chunk['token_count'])
                        
                        if file_chunks:
                            avg_tokens = sum(c['token_count'] for c in file_chunks) / len(file_chunks)
                            sample_keywords = ', '.join(file_chunks[0]['all_keywords'][:3])
                            print(f"   ✅ {file_name}: {len(file_chunks)} chunks (avg: {avg_tokens:.0f} tokens) [{sample_keywords}]")
                        
                    except Exception as e:
                        folder_stats['error_files'] += 1
                        print(f"   ❌ Error: {file_name} - {str(e)[:50]}...")
            
            detailed_stats['processing_stats'][category] = folder_stats
            detailed_stats['processing_stats'][category]['chunks_created'] = len(folder_chunks)
            detailed_stats['processing_stats'][category]['avg_tokens_per_chunk'] = np.mean(chunk_sizes) if chunk_sizes else 0
            detailed_stats['processing_stats'][category]['token_std'] = np.std(chunk_sizes) if chunk_sizes else 0
            
            detailed_stats['keyword_stats'][category] = {
                'unique_keywords': len(folder_keywords),
                'sample_keywords': list(folder_keywords)[:10],
                'avg_keywords_per_chunk': np.mean([len(c['all_keywords']) for c in folder_chunks]) if folder_chunks else 0
            }
            
            detailed_stats['chunk_size_distribution'].extend(chunk_sizes)
            all_chunks.extend(folder_chunks)
            
            print(f"   📊 Summary: {folder_stats['files_processed']} files → {len(folder_chunks)} chunks")
            print(f"   🏷️  Keywords: {len(folder_keywords)} unique, avg {detailed_stats['keyword_stats'][category]['avg_keywords_per_chunk']:.1f} per chunk")
            print()
        
        detailed_stats['quality_metrics'] = self.calculate_quality_metrics(all_chunks)
        self.print_enhanced_summary(detailed_stats, all_chunks)
        
        return all_chunks, detailed_stats
    
    def calculate_quality_metrics(self, chunks: List[Dict]) -> Dict:
        """Calculate quality metrics for chunks"""
        if not chunks:
            return {}
        
        token_counts = [c['token_count'] for c in chunks]
        chunk_densities = [c['chunk_density'] for c in chunks]
        sentence_lengths = [c['avg_sentence_length'] for c in chunks]
        
        return {
            'total_chunks': len(chunks),
            'token_distribution': {
                'mean': np.mean(token_counts),
                'std': np.std(token_counts),
                'min': np.min(token_counts),
                'max': np.max(token_counts),
                'within_target_range': sum(1 for t in token_counts if self.min_chunk_size <= t <= self.max_chunk_size)
            },
            'chunk_quality': {
                'avg_density': np.mean(chunk_densities),
                'avg_sentence_length': np.mean(sentence_lengths),
                'well_formed_chunks': sum(1 for c in chunks if c['sentence_count'] >= 2)
            },
            'keyword_coverage': {
                'avg_keywords_per_chunk': np.mean([len(c['all_keywords']) for c in chunks]),
                'chunks_with_content_keywords': sum(1 for c in chunks if len(c['content_keywords']) > 0)
            }
        }
    
    def print_enhanced_summary(self, stats: Dict, all_chunks: List[Dict]):
        """Print processing summary"""
        print("=" * 70)
        print("📊 PROCESSING SUMMARY")
        print("=" * 70)
        
        processing_stats = stats['processing_stats']
        quality_metrics = stats['quality_metrics']
        
        total_files = sum(s['files_processed'] for s in processing_stats.values())
        total_chunks = len(all_chunks)
        avg_tokens = quality_metrics['token_distribution']['mean']
        
        print(f"📈 Overall Results:")
        print(f"   Files processed: {total_files}")
        print(f"   Chunks created: {total_chunks}")
        print(f"   Average tokens per chunk: {avg_tokens:.1f} ± {quality_metrics['token_distribution']['std']:.1f}")
        print(f"   Chunks in target range: {quality_metrics['token_distribution']['within_target_range']}/{total_chunks} ({quality_metrics['token_distribution']['within_target_range']/total_chunks*100:.1f}%)")
        print()
        
        print(f"📂 By Category:")
        print("-" * 50)
        for category, stat in processing_stats.items():
            kw_stat = stats['keyword_stats'][category]
            print(f"{category:<20} {stat['files_processed']:>3} files → {stat['chunks_created']:>4} chunks")
            print(f"{'':>20} avg: {stat['avg_tokens_per_chunk']:>5.0f} tokens, {kw_stat['unique_keywords']:>3} keywords")
        print()
        
        print(f"⭐ Quality Metrics:")
        print(f"   Well-formed chunks: {quality_metrics['chunk_quality']['well_formed_chunks']}/{total_chunks} ({quality_metrics['chunk_quality']['well_formed_chunks']/total_chunks*100:.1f}%)")
        print(f"   Average chunk density: {quality_metrics['chunk_quality']['avg_density']:.2f}")
        print(f"   Chunks with content keywords: {quality_metrics['keyword_coverage']['chunks_with_content_keywords']}/{total_chunks}")
        print()
        print("✅ Processing complete! Ready for RAG pipeline.")
    
    def save_enhanced_chunks(self, chunks: List[Dict], stats: Dict, output_dir: str = "enhanced_chunks_650"):
        """Save chunks and statistics"""
        os.makedirs(output_dir, exist_ok=True)
        
        chunks_file = os.path.join(output_dir, "enhanced_chunks.json")
        with open(chunks_file, 'w', encoding='utf-8') as f:
            json.dump(chunks, f, indent=2, ensure_ascii=False)
        
        stats_file = os.path.join(output_dir, "processing_statistics.json")
        with open(stats_file, 'w', encoding='utf-8') as f:
            def convert_numpy(obj):
                if isinstance(obj, np.integer):
                    return int(obj)
                elif isinstance(obj, np.floating):
                    return float(obj)
                elif isinstance(obj, np.ndarray):
                    return obj.tolist()
                return obj
            
            stats_serializable = json.loads(json.dumps(stats, default=convert_numpy))
            json.dump(stats_serializable, f, indent=2, ensure_ascii=False)
        
        for chunk in chunks:
            category = chunk['category'].replace(' ', '_').lower()
            category_file = os.path.join(output_dir, f"chunks_{category}.json")
            
            category_chunks = []
            if os.path.exists(category_file):
                with open(category_file, 'r', encoding='utf-8') as f:
                    category_chunks = json.load(f)
            
            category_chunks.append(chunk)
            
            with open(category_file, 'w', encoding='utf-8') as f:
                json.dump(category_chunks, f, indent=2, ensure_ascii=False)
        
        print(f"💾 Chunks saved to: {output_dir}/")
        print(f"   • All chunks: enhanced_chunks.json")
        print(f"   • Statistics: processing_statistics.json") 
        print(f"   • By category: chunks_[category].json")

def main():
    """Main execution function"""
    chunker = EnhancedDocumentChunker(
        model_name="intfloat/e5-base",
        chunk_size=512,
        overlap=100,
        min_chunk_size=200,
        max_chunk_size=800
    )
    
    documents_path = "documents"
    all_chunks, stats = chunker.process_document_folders(documents_path)
    chunker.save_enhanced_chunks(all_chunks, stats)
    
    print("\n" + "=" * 70)
    print("📝 SAMPLE CHUNKS")
    print("=" * 70)
    
    categories_shown = set()
    for chunk in all_chunks[:15]:
        if chunk['category'] not in categories_shown or len(categories_shown) < 3:
            print(f"\n🏷️  Category: {chunk['category']}")
            print(f"📄 File: {chunk['file_name']}")
            print(f"🔤 Size: {chunk['token_count']} tokens | {chunk['word_count']} words | {chunk['sentence_count']} sentences")
            print(f"🏷️  Keywords: {', '.join(chunk['all_keywords'][:5])}")
            print(f"⚖️  Weights: Tech={chunk['technical_weight']}, Narrative={chunk['narrative_weight']}")
            print(f"📊 Quality: Density={chunk['chunk_density']:.2f}, Avg Sent Length={chunk['avg_sentence_length']:.1f}")
            print(f"📜 Preview: {chunk['text'][:250]}...")
            print("-" * 50)
            
            categories_shown.add(chunk['category'])
            
            if len(categories_shown) >= 4:
                break
    
    return all_chunks, stats

if __name__ == "__main__":
    chunks, statistics = main()
    print(f"\n🎉 Processing complete!")
    print(f"Generated {len(chunks)} high-quality chunks optimized for RAG.")
    print("Ready for embedding generation and vector database indexing!")

🚀 Enhanced Document Processing Started
📁 Processing: ICC (icc_text_files)
   ✅ icc01_API_Evolution_Data_Formats_&_The_Emergence_of_Standards_SOAP_&_REST.txt: 1 chunks (avg: 493 tokens) [markup language, xml, formats]
   ✅ icc02_Advantages_r_Disadvantages_of_Private_Cloud_&_Hybrid_Cloud_Introduction.txt: 1 chunks (avg: 481 tokens) [private, hybrid, better control]
   ✅ icc03_Advantages_r_Disadvantages_of_Public_Cloud_&_Private_Cloud_Introduction.txt: 1 chunks (avg: 491 tokens) [private, consequently, minimal investment]
   ✅ icc04_Alternatives_to_VMs_Containers_Introduction.txt: 1 chunks (avg: 499 tokens) [do we have 
an alternative?, vms, icc04]
   ✅ icc05_Basic_Security_Terms_Continued_&_Threat_Agents.txt: 1 chunks (avg: 509 tokens) [security, terms, continued]
   ✅ icc06_Benefits_and_Steps_of_Cloud_Threat_Modeling.txt: 1 chunks (avg: 489 tokens) [cloud threat modeling, high risks, benefits]
   ✅ icc07_Big_Data_Definition_Sources_Examples_&_Four_Dimensions.txt: 1 chunks (avg: 502 toke

Token indices sequence length is longer than the specified maximum sequence length for this model (548 > 512). Running this sequence through the model will result in indexing errors


   ✅ lec3_(n)_Example_of_Confusion_matrix_percision_recall_accuracy_F1_Score.txt: 2 chunks (avg: 527 tokens) [example, email spam detection confusion matrix example, score]
   ✅ lec4-(a) Probability.txt: 2 chunks (avg: 549 tokens) [words, natural language processing, probability]
   ✅ lec4-(b) Conditional Probability.txt: 2 chunks (avg: 536 tokens) [conditional probability, natural language processing, conditional]
   ✅ lec4-(c) Language Models.txt: 2 chunks (avg: 467 tokens) [language models, deeper dive a, language]
   ✅ lec4-(d) Chain Rule of Probability.txt: 3 chunks (avg: 430 tokens) [rule, probability the, probability]
   ✅ lec4-(e) N-gram Model.txt: 2 chunks (avg: 518 tokens) [model, what, unigram]
   ✅ lec4-(f) Markov Assumption.txt: 1 chunks (avg: 494 tokens) [assumption, the markov, for]
   ✅ lec4-(g) Spelling Correction Using N-grams and Edit Distance.txt: 1 chunks (avg: 463 tokens) [spelling correction an, distance, edit]
   ✅ lec4-(h) Google Search Operators.txt: 1 chunks 

# Embeddings & Vector Store

Dataset embeddings were applied using **"intfloat/e5-base"** embedder

FAISS (Facebook AI Similarity Search) vector store was created for similarity-based retrieval

# 🚀 Getting Started

## Setup Instructions
1. Read the `README.md` for environment requirements and setup
2. Download all required models and data as described in README
3. Execute cells sequentially starting from here
4. Ensure all dependencies are installed and files are in correct directories

## Download Embedding Model: intfloat/e5-base

In [ ]:
from transformers import AutoTokenizer, AutoModel
import os
import shutil

from huggingface_hub import snapshot_download

local_model_dir = os.path.join("models", "intfloat__e5-base")

print("📥 Downloading model...")
downloaded_path = snapshot_download(repo_id="intfloat/e5-base")

if not os.path.exists(local_model_dir):
    print(f"📦 Copying model to {local_model_dir} ...")
    shutil.copytree(downloaded_path, local_model_dir)

print(f"✅ Model is saved in {local_model_dir}")


b:\Course RAG project\Course_RAG\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


📥 Downloading model...


Fetching 12 files: 100%|██████████| 12/12 [00:00<00:00, 12061.26it/s]


📦 Copying model to models\intfloat__e5-base ...
✅ Model is saved in models\intfloat__e5-base


## Download LLM Model: Llama-3.2-1B-Instruct

**Download Link:** [Llama-3.2-1B-Instruct-Q4_K_M.gguf](https://huggingface.co/bartowski/Llama-3.2-1B-Instruct-GGUF/blob/main/Llama-3.2-1B-Instruct-Q4_K_M.gguf)

Save the model file in the `models/` directory

# Load Models into Memory

Load embedding and LLM models once to avoid reloading for each query session

In [ ]:
import json
import numpy as np
import faiss
import time
from typing import List, Dict, Any, Tuple
from sentence_transformers import SentenceTransformer
import pickle
import os
from dataclasses import dataclass
from pathlib import Path

try:
    from llama_cpp import Llama
    LLAMA_CPP_AVAILABLE = True
except ImportError:
    LLAMA_CPP_AVAILABLE = False
    print("Warning: llama-cpp-python not installed. LLM generation will be simulated.")

@dataclass
class RAGConfig:
    """Configuration for RAG pipeline"""
    faiss_index_path: str = "faiss_vector_store/faiss_index.index"
    chunk_metadata_path: str = "faiss_vector_store/chunk_metadata.json"
    vector_store_metadata_path: str = "faiss_vector_store/vector_store_metadata.json"
    
    embedding_model_name: str = "intfloat/e5-base"
    llm_model_path: str = "models/llama-3.2-1b-instruct-q4_k_m.gguf"
    
    top_k_dense: int = 3       # how manay files to choose from when looking 
    similarity_threshold: float = 0.4    # atleast 60% matching file will be selceted
    
    max_tokens: int = 1000  # Maximum tokens for LLM response generation
    temperature: float = 0.3  # how much you awnt LLM to add its own understanding 
    top_p: float = 0.85  
    context_length: int = 3500  # Max token limit of generated content

class DenseRetriever:
    """Dense retrieval using FAISS"""
    
    def __init__(self, config: RAGConfig):
        self.config = config
        self.embedding_model = None
        self.faiss_index = None
        self.chunk_metadata = None
        self.vector_store_metadata = None
        
    def initialize(self):
        """Initialize retriever components"""
        print("\n" + "="*80)
        print("🔄 INITIALIZING DENSE RETRIEVER")
        print("="*80)
        
        print("📥 Loading embedding model...")
        start_time = time.time()
        self.embedding_model = SentenceTransformer(self.config.embedding_model_name)
        print(f"✅ Embedding model loaded in {time.time() - start_time:.2f}s")
        
        print("📥 Loading FAISS index...")
        start_time = time.time()
        if not os.path.exists(self.config.faiss_index_path):
            raise FileNotFoundError(f"FAISS index not found: {self.config.faiss_index_path}")
        
        self.faiss_index = faiss.read_index(self.config.faiss_index_path)
        print(f"✅ FAISS index loaded in {time.time() - start_time:.2f}s")
        print(f"📊 Index contains {self.faiss_index.ntotal} vectors")
        
        print("📥 Loading chunk metadata...")
        if not os.path.exists(self.config.chunk_metadata_path):
            raise FileNotFoundError(f"Chunk metadata not found: {self.config.chunk_metadata_path}")
        with open(self.config.chunk_metadata_path, 'r', encoding='utf-8') as f:
            self.chunk_metadata = json.load(f)
        
        if os.path.exists(self.config.vector_store_metadata_path):
            with open(self.config.vector_store_metadata_path, 'r', encoding='utf-8') as f:
                self.vector_store_metadata = json.load(f)
        else:
            self.vector_store_metadata = {}
            
        print(f"✅ Loaded metadata for {len(self.chunk_metadata)} chunks")
        print("🚀 Dense Retriever initialized!")
        
    def embed_query(self, query: str) -> np.ndarray:
        """Embed query text"""
        prefixed_query = f"query: {query}"
        start_time = time.time()
        embedding = self.embedding_model.encode([prefixed_query], normalize_embeddings=True).astype(np.float32)
        embed_time_ms = (time.time() - start_time) * 1000
        print(f"⚡ Query embedded in {embed_time_ms:.1f}ms")
        return embedding[0]
    
    def search_similar_chunks(self, query_embedding: np.ndarray) -> List[Dict[str, Any]]:
        """Search for similar chunks using FAISS"""
        start_time = time.time()
        scores, indices = self.faiss_index.search(
            query_embedding.reshape(1, -1), 
            self.config.top_k_dense
        )
        search_time_ms = (time.time() - start_time) * 1000
        print(f"🔍 FAISS search completed in {search_time_ms:.1f}ms")
        
        results = []
        for i, (score, idx) in enumerate(zip(scores[0], indices[0])):
            if idx < len(self.chunk_metadata) and score >= self.config.similarity_threshold:
                chunk_data = self.chunk_metadata[idx].copy()
                chunk_data['similarity_score'] = float(score)
                chunk_data['retrieval_rank'] = i + 1
                results.append(chunk_data)
            elif score < self.config.similarity_threshold:
                break
        
        print(f"📋 Retrieved {len(results)} relevant chunks (threshold: {self.config.similarity_threshold:.2f})")
        return results
    
    def retrieve(self, query: str) -> List[Dict[str, Any]]:
        """Main retrieval method"""
        print(f"\n" + "-"*80)
        print(f"🔍 RETRIEVAL FOR: '{query[:100]}{'...' if len(query) > 100 else ''}'")
        print("-"*80)
        
        query_embedding = self.embed_query(query)
        results = self.search_similar_chunks(query_embedding)
        
        if results:
            print(f"\n📊 TOP {len(results)} RETRIEVED CHUNKS:")
            for i, result_chunk in enumerate(results):
                file_name = Path(result_chunk.get('source_file', 'Unknown')).name
                print(f"  {result_chunk['retrieval_rank']}. {file_name} (Score: {result_chunk['similarity_score']:.4f})")
        else:
            print("\n⚠️ No chunks retrieved")

        return results

class LLMGenerator:
    """LLM generation using LLaMA 3.2-1B"""
    
    def __init__(self, config: RAGConfig):
        self.config = config
        self.llm = None
        
    def initialize(self):
        """Initialize LLM"""
        print("\n" + "="*80)
        print("🤖 INITIALIZING LLM GENERATOR")
        print("="*80)
        
        if not LLAMA_CPP_AVAILABLE:
            print("⚠️  llama-cpp-python not available. Using simulation mode.")
            return
            
        if not os.path.exists(self.config.llm_model_path):
            print(f"⚠️  LLM model not found: {self.config.llm_model_path}")
            print("   Using simulation mode.")
            return
        
        print(f"📥 Loading LLaMA GGUF model...")
        start_time = time.time()
        
        try:
            self.llm = Llama(
                model_path=self.config.llm_model_path,
                n_ctx=self.config.context_length,
                n_threads=os.cpu_count(),
                verbose=False,
                n_gpu_layers=-1
            )
            print(f"✅ LLM loaded in {time.time() - start_time:.2f}s")
            print("🚀 LLM Generator initialized!")
        except Exception as e:
            print(f"❌ Error loading LLM: {e}")
            print("   Using simulation mode.")
            self.llm = None
    
    def format_prompt(self, query: str, retrieved_chunks: List[Dict[str, Any]]) -> str:
        """Format prompt with retrieved context"""
        context_parts = []
        for i, chunk in enumerate(retrieved_chunks):
            source_file_name = Path(chunk.get('source_file', 'Unknown')).name
            text = chunk.get('text', '')
            score = chunk.get('similarity_score', 0.0)
            context_parts.append(
                f"<document id={i+1} source={source_file_name} relevance={score:.3f}>\n{text}\n</document>"
            )
        
        context_string = "\n".join(context_parts)
        
        prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are an AI assistant that provides comprehensive answers using provided context documents and your knowledge.

INSTRUCTIONS:
1. Use information from context documents as primary source
2. Cite sources using [Document X: filename.ext]
3. Be comprehensive - supplement with knowledge if documents have partial info
4. If NO relevant documents, respond: "NO RELEVANT INFORMATION FOUND in the provided documents for this query."
5. Synthesize information from multiple documents when applicable
6. Be helpful and accurate

<|eot_id|><|start_header_id|>user<|end_header_id|>

Context Documents:
{context_string}

Question: {query}

Provide a comprehensive answer.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

"""
        return prompt
    
    def generate_response(self, query: str, retrieved_chunks: List[Dict[str, Any]]) -> Dict[str, Any]:
        """Generate response using LLM"""
        print(f"\n" + "-"*80)
        print(f"🤖 GENERATION PHASE")
        print("-"*80)
        
        if not retrieved_chunks:
            print("⚠️ No relevant chunks retrieved.")
            return {
                'response': "NO RELEVANT INFORMATION FOUND in the provided documents for this query.",
                'sources': [],
                'generation_time': 0,
                'token_count': 0,
                'simulated': True,
                'no_context': True
            }
        
        prompt = self.format_prompt(query, retrieved_chunks)
        
        if self.llm is None:
            print("🔄 Simulating LLM response...")
            time.sleep(1.5)
            
            sources_list = sorted(list(set(Path(chunk.get('source_file', 'Unknown')).name for chunk in retrieved_chunks)))
            
            simulated_response = f"""Based on retrieved documents: {', '.join(sources_list[:3])}...

**Sources**: {len(retrieved_chunks)} relevant chunks found.

[SIMULATED - Install llama-cpp-python and LLaMA model for actual AI responses]

**Answer**: [The LLM would provide a comprehensive synthesized answer here]
"""
            print(f"✅ Simulated response generated")
            return {
                'response': simulated_response,
                'sources': sources_list,
                'generation_time': 1.5,
                'token_count': len(simulated_response.split()),
                'simulated': True
            }
        
        print(f"🧠 Calling LLaMA for generation...")
        start_time = time.time()
        
        try:
            output = self.llm.create_completion(
                prompt,
                max_tokens=self.config.max_tokens,
                temperature=self.config.temperature,
                top_p=self.config.top_p,
                stop=["<|eot_id|>"],
                echo=False
            )
            
            generation_time = time.time() - start_time
            response_text = output['choices'][0]['text'].strip()
            sources_list = sorted(list(set(Path(chunk.get('source_file', 'Unknown')).name for chunk in retrieved_chunks)))
            
            print(f"✅ Response generated in {generation_time:.2f}s")
            print(f"📊 Tokens: {output['usage']['completion_tokens']}")
            
            return {
                'response': response_text,
                'sources': sources_list,
                'generation_time': generation_time,
                'token_count': output['usage']['completion_tokens'],
                'simulated': False
            }
            
        except Exception as e:
            print(f"❌ Error during generation: {e}")
            return {
                'response': f"Error during generation: {str(e)}",
                'sources': [],
                'generation_time': 0,
                'token_count': 0,
                'error': True,
                'simulated': False
            }

class RAGPipeline:
    """Complete RAG pipeline"""
    
    def __init__(self, config: RAGConfig = None):
        self.config = config or RAGConfig()
        self.retriever = DenseRetriever(self.config)
        self.generator = LLMGenerator(self.config)
        
    def initialize(self):
        """Initialize retriever and generator"""
        print("\n" + "="*80)
        print("🚀 INITIALIZING RAG PIPELINE")
        print("="*80)
        self.retriever.initialize()
        self.generator.initialize()
        print("\n" + "="*80)
        print("✅ RAG PIPELINE READY!")
        print("="*80)
    
    def save_pipeline(self, filepath: str = "rag_pipeline.pkl"):
        """Save pipeline to file"""
        print(f"\n💾 Saving pipeline to {filepath}...")
        try:
            with open(filepath, 'wb') as f:
                pickle.dump(self, f)
            print(f"✅ Pipeline saved!")
        except Exception as e:
            print(f"❌ Error saving: {e}")
    
    @classmethod
    def load_pipeline(cls, filepath: str = "rag_pipeline.pkl"):
        """Load pipeline from file"""
        print(f"\n📥 Loading pipeline from {filepath}...")
        try:
            with open(filepath, 'rb') as f:
                pipeline = pickle.load(f)
            print(f"✅ Pipeline loaded!")
            return pipeline
        except Exception as e:
            print(f"❌ Error loading: {e}")
            return None
        
    def query(self, question: str) -> Dict[str, Any]:
        """Process RAG query"""
        start_time = time.time()
        
        retrieved_chunks = self.retriever.retrieve(question)
        retrieval_time = time.time() - start_time
        
        generation_start = time.time()
        result = self.generator.generate_response(question, retrieved_chunks)
        generation_time = time.time() - generation_start
        
        total_time = time.time() - start_time
        
        return {
            'query': question,
            'retrieved_chunks_count': len(retrieved_chunks),
            'retrieval_time': retrieval_time,
            'generation_time': generation_time,
            'total_time': total_time,
            'response': result['response'],
            'sources': result['sources'],
            'chunk_details': retrieved_chunks,
            'no_context': result.get('no_context', False)
        }

def setup_rag_pipeline():
    """Setup and initialize RAG pipeline"""
    print("="*80)
    print("🚀 RAG PIPELINE SETUP")
    print("="*80)
    
    config = RAGConfig()
    rag = RAGPipeline(config)
    
    try:
        rag.initialize()
        rag.save_pipeline()
        
        print("\n" + "="*80)
        print("✅ SETUP COMPLETE!")
        print("Pipeline initialized and saved.")
        print("="*80)
        
        return rag
        
    except Exception as e:
        print(f"\n❌ Setup error: {e}")
        import traceback
        traceback.print_exc()
        return None

if __name__ == "__main__":
    setup_rag_pipeline()

🚀 RAG PIPELINE SETUP AND INITIALIZATION

🚀 INITIALIZING COMPLETE RAG PIPELINE

🔄 INITIALIZING DENSE RETRIEVER
📥 Loading embedding model (this may take a moment, downloads if not cached)...
✅ Embedding model loaded in 4.34 seconds
📥 Loading FAISS index...
✅ FAISS index loaded in 0.04 seconds
📊 Index contains 704 vectors
📥 Loading chunk metadata...
📥 Loading vector store metadata...
✅ Loaded metadata for 704 chunks
🚀 Dense Retriever initialized successfully!

🤖 INITIALIZING LLM GENERATOR
📥 Loading LLaMA 3.2-1B GGUF model (this can take several seconds)...


llama_context: n_ctx_per_seq (3500) < n_ctx_train (131072) -- the full capacity of the model will not be utilized


✅ LLM loaded in 2.39 seconds
🚀 LLM Generator initialized successfully!

✅ RAG PIPELINE READY!

💾 Saving RAG pipeline to rag_pipeline.pkl...
✅ RAG pipeline saved successfully!

✅ SETUP COMPLETE!
The RAG pipeline has been initialized and saved.
You can now run the query session script.


## Query Session Instructions

Response time: approximately 50 seconds per query

**Note:** For long outputs, use the scrollable element in the output cell or copy to an external editor

Type `exit` to stop the session

# Query Session: Max Tokens 1000

In [ ]:
import pickle
import time
from pathlib import Path
from typing import Dict, Any

def load_rag_pipeline(filepath: str = "rag_pipeline.pkl"):
    """Load pre-initialized RAG pipeline"""
    print(f"\n📥 Loading pipeline from {filepath}...")
    try:
        with open(filepath, 'rb') as f:
            pipeline = pickle.load(f)
        print(f"✅ Pipeline loaded!")
        return pipeline
    except FileNotFoundError:
        print(f"❌ Pipeline file not found at {filepath}")
        print("Run the setup script first.")
        return None
    except Exception as e:
        print(f"❌ Error loading: {e}")
        return None

def display_query_result(result: Dict[str, Any]):
    """Display query results"""
    print(f"\n" + "="*80)
    print(f"QUERY: '{result['query']}'")
    print("="*80)
    
    print(f"\n📝 Response:")
    print("-" * 70)
    print(result['response'])
    print("-" * 70)
    
    if result['chunk_details']:
        print(f"\n📚 Retrieved Documents (Top {result['retrieved_chunks_count']}):")
        print("-" * 70)
        for j, chunk in enumerate(result['chunk_details']):
            file_name = Path(chunk.get('source_file', 'Unknown')).name 
            print(f"  {j+1}. {file_name}")
            print(f"     Category: {chunk.get('category', 'Unknown')}")
            print(f"     Relevance: {chunk.get('similarity_score', 0):.4f}")
            print("-" * 70)
    else:
        print("\n⚠️ No relevant documents retrieved.")
    
    print(f"\n📊 Metrics:")
    print(f"  • Retrieval: {result['retrieval_time']:.3f}s")
    print(f"  • Generation: {result['generation_time']:.3f}s")
    print(f"  • Total: {result['total_time']:.3f}s")
    print("\n" + "="*80 + "\n")

def run_interactive_session(rag_pipeline):
    """Run interactive query session"""
    print("\n" + "="*80)
    print("🚀 RAG PIPELINE INTERACTIVE SESSION")
    print("Type 'exit' or 'quit' to end.")
    print("="*80)
    
    while True:
        print("\n" + "-"*50)
        user_query = input("🤔 Your question: ").strip()
        
        if user_query.lower() in ['exit', 'quit']:
            print("\nGoodbye! 👋")
            break
            
        if not user_query:
            print("⚠️ Please enter a question.")
            continue
            
        try:
            print("\n🔄 Processing...")
            result = rag_pipeline.query(user_query)
            
            print(f"\n💡 **Answer:**")
            print("-" * 50)
            print(result['response'])
            print("-" * 50)
            
            if result['sources']:
                print(f"\n📚 **Sources:** {', '.join(result['sources'])}")
            
            print(f"⏱️ **Time:** {result['total_time']:.2f}s")
            
        except Exception as e:
            print(f"❌ Error: {e}")

def run_single_query(rag_pipeline, query: str):
    """Run single query"""
    print(f"\n🔍 Processing: '{query}'")
    
    try:
        result = rag_pipeline.query(query)
        display_query_result(result)
        return result
    except Exception as e:
        print(f"❌ Error: {e}")
        return None

def main():
    """Main query session"""
    print("="*80)
    print("🚀 RAG PIPELINE QUERY SESSION")
    print("="*80)
    
    rag_pipeline = load_rag_pipeline()
    
    if rag_pipeline is None:
        print("\n❌ Could not load pipeline.")
        return
    
    print("\n✅ Pipeline ready!")
    run_interactive_session(rag_pipeline)

if __name__ == "__main__":
    main()

🚀 RAG PIPELINE QUERY SESSION

📥 Loading RAG pipeline from rag_pipeline.pkl...


llama_context: n_ctx_per_seq (3500) < n_ctx_train (131072) -- the full capacity of the model will not be utilized


✅ RAG pipeline loaded successfully!

✅ RAG pipeline loaded and ready!

🚀 RAG PIPELINE INTERACTIVE SESSION
Type 'exit' or 'quit' to end the session.

--------------------------------------------------

🔄 Processing your question...

--------------------------------------------------------------------------------
🔍 RETRIEVAL PHASE FOR QUERY: 'Define Vector Model, its steps and explain with example'
--------------------------------------------------------------------------------
⚡ Query embedded in 160.1ms
🔍 FAISS search completed in 1.0ms
📋 Retrieved 3 relevant chunks (min threshold: 0.40)

📊 TOP 3 RETRIEVED CHUNKS (Summary):
  1. File: lec6-(f) Vector Space Models (VSMs).txt (Score: 0.8610)
  2. File: lec6-(f) Vector Space Models (VSMs).txt (Score: 0.8488)
  3. File: lec6-(a) Similarity Measure.txt (Score: 0.8303)

--------------------------------------------------------------------------------
🤖 GENERATION PHASE
--------------------------------------------------------------------------

# Query Session: Max Tokens 756

In [ ]:
import pickle
import time
from pathlib import Path
from typing import Dict, Any

def load_rag_pipeline(filepath: str = "rag_pipeline.pkl"):
    """Load pre-initialized RAG pipeline"""
    print(f"\n📥 Loading pipeline from {filepath}...")
    try:
        with open(filepath, 'rb') as f:
            pipeline = pickle.load(f)
        print(f"✅ Pipeline loaded!")
        return pipeline
    except FileNotFoundError:
        print(f"❌ Pipeline file not found")
        return None
    except Exception as e:
        print(f"❌ Error loading: {e}")
        return None

def display_query_result(result: Dict[str, Any]):
    """Display query results"""
    print(f"\n" + "="*80)
    print(f"QUERY: '{result['query']}'")
    print("="*80)
    
    print(f"\n📝 Response:")
    print("-" * 70)
    print(result['response'])
    print("-" * 70)
    
    if result['chunk_details']:
        print(f"\n📚 Retrieved Documents:")
        print("-" * 70)
        for j, chunk in enumerate(result['chunk_details']):
            file_name = Path(chunk.get('source_file', 'Unknown')).name 
            print(f"  {j+1}. {file_name}")
            print(f"     Category: {chunk.get('category', 'Unknown')}")
            print(f"     Relevance: {chunk.get('similarity_score', 0):.4f}")
            print("-" * 70)
    else:
        print("\n⚠️ No documents retrieved.")
    
    print(f"\n📊 Metrics:")
    print(f"  • Retrieval: {result['retrieval_time']:.3f}s")
    print(f"  • Generation: {result['generation_time']:.3f}s")
    print(f"  • Total: {result['total_time']:.3f}s")
    print("\n" + "="*80 + "\n")

def run_interactive_session(rag_pipeline):
    """Interactive query session"""
    print("\n" + "="*80)
    print("🚀 RAG PIPELINE SESSION")
    print("Type 'exit' or 'quit' to end.")
    print("="*80)
    
    while True:
        print("\n" + "-"*50)
        user_query = input("🤔 Your question: ").strip()
        
        if user_query.lower() in ['exit', 'quit']:
            print("\nGoodbye! 👋")
            break
            
        if not user_query:
            print("⚠️ Please enter a question.")
            continue
            
        try:
            print("\n🔄 Processing...")
            result = rag_pipeline.query(user_query)
            
            print(f"\n💡 **Answer:**")
            print("-" * 50)
            print(result['response'])
            print("-" * 50)
            
            if result['sources']:
                print(f"\n📚 **Sources:** {', '.join(result['sources'])}")
            
            print(f"⏱️ **Time:** {result['total_time']:.2f}s")
            
        except Exception as e:
            print(f"❌ Error: {e}")

def main():
    """Main query session"""
    print("="*80)
    print("🚀 RAG PIPELINE QUERY SESSION")
    print("="*80)
    
    rag_pipeline = load_rag_pipeline()
    
    if rag_pipeline is None:
        print("\n❌ Could not load pipeline.")
        return
    
    print("\n✅ Pipeline ready!")
    run_interactive_session(rag_pipeline)

if __name__ == "__main__":
    main()

🚀 RAG PIPELINE QUERY SESSION

📥 Loading RAG pipeline from rag_pipeline.pkl...


llama_context: n_ctx_per_seq (3500) < n_ctx_train (131072) -- the full capacity of the model will not be utilized


✅ RAG pipeline loaded successfully!

✅ RAG pipeline loaded and ready!

🚀 RAG PIPELINE INTERACTIVE SESSION
Type 'exit' or 'quit' to end the session.

--------------------------------------------------

🔄 Processing your question...

--------------------------------------------------------------------------------
🔍 RETRIEVAL PHASE FOR QUERY: 'Define confusion matrix, accuracy, recall, precision and F1 score their formulas and explain example'
--------------------------------------------------------------------------------
⚡ Query embedded in 2778.4ms
🔍 FAISS search completed in 147.4ms
📋 Retrieved 3 relevant chunks (min threshold: 0.40)

📊 TOP 3 RETRIEVED CHUNKS (Summary):
  1. File: lec3-(g) Confusion Matrix.txt (Score: 0.8645)
  2. File: lec3-(m) Multi-class Confusion Matrix.txt (Score: 0.8560)
  3. File: lec3-(i) Precision vs recall.txt (Score: 0.8443)

--------------------------------------------------------------------------------
🤖 GENERATION PHASE
---------------------------------

b:\Course RAG project\Course_RAG\Lib\site-packages\llama_cpp\llama.py:1240: RuntimeWarning: Detected duplicate leading "<|begin_of_text|>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(


✅ LLM Response generated in 155.63 seconds
📊 Tokens generated: 756

💡 **Answer:**
--------------------------------------------------
Here's a comprehensive explanation of confusion matrix, accuracy, recall, precision, and F1 score, along with examples:

**Confusion Matrix:**

A confusion matrix is a table used to evaluate the performance of a classification model. It's a 2x2 matrix, where the rows represent the actual classes, and the columns represent the predicted classes. The matrix has four cells:

* TP (True Positive): The model correctly predicted the positive class.
* FP (False Positive): The model predicted a positive class but the actual class is negative.
* FN (False Negative): The model predicted a negative class but the actual class is positive.
* TN (True Negative): The model correctly predicted the negative class.

The confusion matrix helps calculate evaluation metrics like accuracy, precision, and recall.

**Accuracy:**

Accuracy is the proportion of correctly classifie

In [ ]:
import pickle
import time
from pathlib import Path
from typing import Dict, Any

def load_rag_pipeline(filepath: str = "rag_pipeline.pkl"):
    """Load pre-initialized RAG pipeline"""
    print(f"\n📥 Loading pipeline...")
    try:
        with open(filepath, 'rb') as f:
            pipeline = pickle.load(f)
        print(f"✅ Loaded!")
        return pipeline
    except FileNotFoundError:
        print(f"❌ File not found")
        return None
    except Exception as e:
        print(f"❌ Error: {e}")
        return None

def run_interactive_session(rag_pipeline):
    """Interactive query session"""
    print("\n" + "="*80)
    print("🚀 RAG PIPELINE SESSION")
    print("Type 'exit' to end.")
    print("="*80)
    
    while True:
        print("\n" + "-"*50)
        user_query = input("🤔 Question: ").strip()
        
        if user_query.lower() in ['exit', 'quit']:
            print("\nGoodbye! 👋")
            break
            
        if not user_query:
            print("⚠️ Enter a question.")
            continue
            
        try:
            print("\n🔄 Processing...")
            result = rag_pipeline.query(user_query)
            
            print(f"\n💡 **Answer:**")
            print("-" * 50)
            print(result['response'])
            print("-" * 50)
            
            if result['sources']:
                print(f"\n📚 **Sources:** {', '.join(result['sources'])}")
            
            print(f"⏱️ **Time:** {result['total_time']:.2f}s")
            
        except Exception as e:
            print(f"❌ Error: {e}")

def main():
    """Main session"""
    print("="*80)
    print("🚀 RAG QUERY SESSION")
    print("="*80)
    
    rag_pipeline = load_rag_pipeline()
    
    if rag_pipeline is None:
        print("\n❌ Could not load pipeline.")
        return
    
    print("\n✅ Ready!")
    run_interactive_session(rag_pipeline)

if __name__ == "__main__":
    main()

🚀 RAG PIPELINE QUERY SESSION

📥 Loading RAG pipeline from rag_pipeline.pkl...


llama_context: n_ctx_per_seq (3500) < n_ctx_train (131072) -- the full capacity of the model will not be utilized


✅ RAG pipeline loaded successfully!

✅ RAG pipeline loaded and ready!

🚀 RAG PIPELINE INTERACTIVE SESSION
Type 'exit' or 'quit' to end the session.

--------------------------------------------------

🔄 Processing your question...

--------------------------------------------------------------------------------
🔍 RETRIEVAL PHASE FOR QUERY: 'Explain Vsm model and its steps'
--------------------------------------------------------------------------------
⚡ Query embedded in 127.3ms
🔍 FAISS search completed in 1.0ms
📋 Retrieved 3 relevant chunks (min threshold: 0.40)

📊 TOP 3 RETRIEVED CHUNKS (Summary):
  1. File: lec6-(f) Vector Space Models (VSMs).txt (Score: 0.8611)
  2. File: lec6-(f) Vector Space Models (VSMs).txt (Score: 0.8274)
  3. File: icc06_Benefits_and_Steps_of_Cloud_Threat_Modeling.txt (Score: 0.8187)

--------------------------------------------------------------------------------
🤖 GENERATION PHASE
----------------------------------------------------------------------------